# Week 6 — Hands-On: Neural Networks with PyTorch

Build, train, and evaluate a multi-layer perceptron (MLP) classifier from scratch using PyTorch.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

## 1. Dataset — Digits (8×8 images of hand-written digits)

In [ ]:
digits = load_digits()
X, y = digits.data.astype(np.float32), digits.target

print(f'Samples: {X.shape[0]}, Features: {X.shape[1]}, Classes: {len(np.unique(y))}')

# Visualise a few samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for ax, img, lbl in zip(axes.flat, digits.images[:10], digits.target[:10]):
    ax.imshow(img, cmap='gray_r')
    ax.set_title(str(lbl))
    ax.axis('off')
plt.suptitle('Sample Digits')
plt.tight_layout()
plt.show()

In [ ]:
X_train_np, X_test_np, y_train_np, y_test_np = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_np = scaler.fit_transform(X_train_np).astype(np.float32)
X_test_np  = scaler.transform(X_test_np).astype(np.float32)

# Convert to tensors
X_train_t = torch.tensor(X_train_np).to(device)
y_train_t = torch.tensor(y_train_np, dtype=torch.long).to(device)
X_test_t  = torch.tensor(X_test_np).to(device)
y_test_t  = torch.tensor(y_test_np, dtype=torch.long).to(device)

train_ds = TensorDataset(X_train_t, y_train_t)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)

## 2. Define the Model

In [ ]:
class MLP(nn.Module):
    def __init__(self, n_in: int, n_hidden: int, n_out: int, dropout: float = 0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_in, n_hidden),
            nn.BatchNorm1d(n_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden, n_hidden // 2),
            nn.BatchNorm1d(n_hidden // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(n_hidden // 2, n_out),
        )

    def forward(self, x):
        return self.net(x)


model = MLP(n_in=64, n_hidden=128, n_out=10).to(device)
print(model)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters: {total_params:,}')

## 3. Train

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.5)

NUM_EPOCHS = 60
train_losses, val_accs = [], []

for epoch in range(1, NUM_EPOCHS + 1):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch), y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(y_batch)
    scheduler.step()

    # Validation
    model.eval()
    with torch.no_grad():
        logits = model(X_test_t)
        preds  = logits.argmax(dim=1).cpu().numpy()
        acc    = accuracy_score(y_test_np, preds)

    train_losses.append(epoch_loss / len(train_ds))
    val_accs.append(acc)

    if epoch % 10 == 0:
        print(f'Epoch {epoch:3d}/{NUM_EPOCHS}  loss={train_losses[-1]:.4f}  val_acc={acc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(train_losses)
axes[0].set_title('Training Loss')
axes[0].set_xlabel('Epoch')
axes[1].plot(val_accs, color='orange')
axes[1].set_title('Validation Accuracy')
axes[1].set_xlabel('Epoch')
plt.tight_layout()
plt.show()

print(f'Final test accuracy: {val_accs[-1]:.4f}')

## 4. Exercises

1. Remove `BatchNorm1d` layers and retrain. Does accuracy drop?
2. Increase dropout to `0.5`. What happens to the training loss vs validation accuracy?
3. Try `optim.SGD` with momentum `0.9` instead of Adam. Compare convergence speed.
4. *Bonus*: reshape each image to 8×8 and build a `Conv2d`-based model using PyTorch.